In [11]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

class HelloAgentLLM:
    def __init__(self, model:str = None, apikey: str = None, baseurl: str = None, timeout:int = None):
        self.model = model or os.getenv("LLM_MODEL_ID")
        apikey = apikey or os.getenv("DEEPSEEK_API_KEY")
        baseurl = baseurl or os.getenv("DEEPSEEK_BASE_URL")
        timeout = timeout or int(os.getenv("LLM_TIMEOUT", 60))

        if not all([self.model, apikey, baseurl]):
            raise ValueError("模型基础配置缺失")
        
        self.client = OpenAI(
            api_key=apikey,
            base_url=baseurl,
            timeout=timeout
        )
    
    def think(self, message: list[dict[str, str]], temproture:float = 0) -> str:
        print(f"正在调用{self.model}模型")

        try:
            response = self.client.chat.completions.create(
                model = self.model,
                messages=message,
                temperature=temproture,
                stream=True
            )
            print("模型响应成功")
            collected_content = []
            for chunk in response:
                content = chunk.choices[0].delta.content or ""
                print(content, end="", flush=True)
                collected_content.append(content)
            print()
            return "".join(collected_content)
        except Exception as e:
            print(f"调用LLM API时发生错误:{e}")
            return None
        
llmClient = HelloAgentLLM(model="deepseek-chat")

PLANNER_PROMPT_TEMPLATE = """
你是一个顶级的AI规划专家。你的任务是将用户提出的复杂问题分解成一个由多个简单步骤组成的行动计划。
请确保计划中的每个步骤都是一个独立的、可执行的子任务，并且严格按照逻辑顺序排列。
你的输出必须是一个Python列表，其中每个元素都是一个描述子任务的字符串。

问题: {question}

请严格按照以下格式输出你的计划,```python与```作为前后缀是必要的:
```python
["步骤1", "步骤2", "步骤3", ...]
```
"""

In [12]:
import ast
class Planner:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def plan(self, question:str)->list[str]:
        prompt = PLANNER_PROMPT_TEMPLATE.format(question=question)
        message = [{"role":"user", "content":prompt}]
        print("--已生成计划--")
        response_text = self.llm_client.think(message=message) or ""
        print(f"计划已生成:\n{response_text}")

        try:
            plan_str = response_text.split("```python")[1].split("```")[0].strip()
            plan = ast.literal_eval(plan_str)
            return plan if isinstance(plan, list) else []
        except (ValueError, SyntaxError, IndexError) as e:
            print(f"解析计划出错{e}")
            print(f"原始响应{response_text}")
            return []
        except Exception as e:
            print(f"未知错误{e}")
            return []


In [13]:
EXECUTOR_PROMPT_TEMPLATE = """
你是一位顶级的AI执行专家。你的任务是严格按照给定的计划，一步步地解决问题。
你将收到原始问题、完整的计划、以及到目前为止已经完成的步骤和结果。
请你专注于解决“当前步骤”，并仅输出该步骤的最终答案，不要输出任何额外的解释或对话。

# 原始问题:
{question}

# 完整计划:
{plan}

# 历史步骤与结果:
{history}

# 当前步骤:
{current_step}

请仅输出针对“当前步骤”的回答:
"""

class Executor:
    def __init__(self, llm_client):
        self.llm_client = llm_client
    
    def excute(self, question: str, plan: list[str])-> str:
        history = ""

        print("--正在执行计划--")
        
        for i, step in enumerate(plan):
            print(f"\n->正在执行步骤{(i+1)}/{len(plan)}:{step}")

            prompt = EXECUTOR_PROMPT_TEMPLATE.format(
                question = question,
                plan=plan,
                history = history if history else "无",
                current_step = step
            )

            message = [{"role":"user", "content":prompt}]

            response_text = self.llm_client.think(message = message)
            history += f"步骤：{i+1}:{step}\n结果：{response_text}\n\n"
            print(f"步骤{i+1}已完成，结果：{response_text}")
        final_answer = response_text
        return final_answer

class PlanAndSolveAgent:
    def __init__(self, llm_client):
        self.llm_client = llm_client
        self.planner = Planner(self.llm_client)
        self.executor = Executor(self.llm_client)
    def run(self, question:str):
        print(f"\n--开始处理问题--\n问题：{question}")

        plan = self.planner.plan(question)

        if not plan:
            print("\n---任务终止---\n无法生成有效计划。")
            return
        final_answer = self.executor.excute(question, plan)

        print(f"\n---任务完成---\n最终答案：{final_answer}")

agent = PlanAndSolveAgent(llm_client=llmClient)
question = "一个水果店周一卖出了15个苹果。周二卖出的苹果数量是周一的两倍。周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？"
agent.run(question=question)


--开始处理问题--
问题：一个水果店周一卖出了15个苹果。周二卖出的苹果数量是周一的两倍。周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？
--已生成计划--
正在调用deepseek-chat模型
模型响应成功
```python
["计算周二卖出的苹果数量：周一卖出的15个苹果乘以2", "计算周三卖出的苹果数量：周二卖出的数量减去5个", "将周一、周二、周三卖出的苹果数量相加，得到三天的总销量"]
```
计划已生成:
```python
["计算周二卖出的苹果数量：周一卖出的15个苹果乘以2", "计算周三卖出的苹果数量：周二卖出的数量减去5个", "将周一、周二、周三卖出的苹果数量相加，得到三天的总销量"]
```
--正在执行计划--

->正在执行步骤1/3:计算周二卖出的苹果数量：周一卖出的15个苹果乘以2
正在调用deepseek-chat模型
模型响应成功
30
步骤1已完成，结果：30

->正在执行步骤2/3:计算周三卖出的苹果数量：周二卖出的数量减去5个
正在调用deepseek-chat模型
模型响应成功
25
步骤2已完成，结果：25

->正在执行步骤3/3:将周一、周二、周三卖出的苹果数量相加，得到三天的总销量
正在调用deepseek-chat模型
模型响应成功
70
步骤3已完成，结果：70

---任务完成---
最终答案：70
